# Clustering and PCA — Solution

**Dataset:** Car models described by engine size, fuel consumption and CO₂ emissions; no target is used.

Run from top to bottom. The steps match the exercise; short answers explain the results.

## Libraries and settings

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

## 1. Move a cluster centre

For points (1,1) and (3,1), replace the gap with `points.mean(axis=0)`. What is the new centre and WCSS?

In [ ]:
points = np.array([[1, 1], [3, 1]])
centre = points.mean(axis=0)
print('New centre:', centre)
print('WCSS:', np.sum((points - centre) ** 2))

**Answer:** The new centre is (2, 1), and WCSS = 2.

- Centre: ((1 + 3) / 2, (1 + 1) / 2) = (2, 1).
- Point (1, 1) has squared distance (1 − 2)² + 0² = 1 to the centre; point (3, 1) has (3 − 2)² + 0² = 1.
- WCSS = 1 + 1 = 2.

## 2. Prepare and scale car features

Preparation is provided. Are any of these variables a target? Why scale before grouping?

In [ ]:
cars = pd.read_csv('../00_Data/car_fuel_consumption_2025.csv')
features = ['engine_size', 'combined', 'co2']
cars = cars.dropna(subset=features).copy()
X = cars[features]
scaler = StandardScaler()
scaled = scaler.fit_transform(X)
print(X.head())
print('Scaled rows:')
print(np.round(scaled[:5], 2))

**Answer:** None of the variables is a target. All three are inputs, and the goal is to find groups, not to predict.

- The variables have different units: engine size in litres (about 1–8), consumption in L/100 km (about 4–25) and CO₂ in g/km (about 100–600).
- Without scaling, CO₂ would dominate every distance, simply because its numbers are larger.
- After scaling, each variable has mean 0 and standard deviation 1. For example, the first car's engine size of 1.5 L becomes −1.26, i.e. 1.26 standard deviations below the average engine size.

## 3. Fit three clusters

Replace the gap with `KMeans(n_clusters=3, n_init=10, random_state=42)`. Look at the plot: are groups perfectly separated?

In [ ]:
kmeans = KMeans(n_clusters=3, n_init=10, random_state=42)
labels = kmeans.fit_predict(scaled)
cars['cluster'] = labels
plt.scatter(cars['engine_size'], cars['combined'], c=labels, cmap='viridis', alpha=0.5)
plt.xlabel('Engine size [L]')
plt.ylabel('Fuel consumption [L/100 km]')
plt.title('Car models, coloured by cluster')
plt.show()

**Answer:** No, the groups are not perfectly separated.

- The clusters form three bands from small, economical cars (bottom left) to large, thirsty cars (top right).
- The clusters touch and overlap, for example around 2–3.5 L engine size and 9–11 L/100 km: cars with the same engine size can belong to different clusters.
- The overlap arises partly because the plot shows only two of the three inputs; CO₂ is not shown.
- k-means always produces the requested number of groups. Three clusters are an illustrative starting choice, not a known true classification.

## 4. Read an elbow plot

The short loop is provided. Where do improvements become smaller? Why is the smallest WCSS alone not a good rule for choosing k?

In [ ]:
ks = range(1, 7)
wcss = []
for k in ks:
    candidate = KMeans(n_clusters=k, n_init=10, random_state=42)
    candidate.fit(scaled)
    wcss.append(candidate.inertia_)
plt.plot(list(ks), wcss, marker='o')
plt.xlabel('Number of clusters k')
plt.ylabel('Within-cluster sum of squares')
plt.show()

**Answer:** The improvements become much smaller after k = 3.

- WCSS for k = 1 to 6: about 2103, 870, 514, 392, 306, 258.
- From k = 1 to 2, WCSS drops by about 1233; from 2 to 3, by 356; from 3 to 4, only by 122; then by 86 and 48.
- The curve bends ("elbow") at k = 2 to 3, so k = 3 is a reasonable choice.
- WCSS always falls when clusters are added; with one cluster per car, it would be 0. Choosing the smallest WCSS would therefore always pick the largest k.
- The chosen groups must also be interpretable and useful.

## 5. Describe the clusters

The table is supplied. Give each cluster a description based on its average engine size, consumption and CO₂. Does cluster 0 mean “best”?

In [ ]:
print('Cluster sizes:')
print(cars['cluster'].value_counts().sort_index())
print('Mean values in original units:')
print(cars.groupby('cluster')[features].mean().round(2))
print('Cluster centres in original units:')
print(pd.DataFrame(scaler.inverse_transform(kmeans.cluster_centers_), columns=features).round(2))

**Answer:**

- **Cluster 2** (303 cars): small, economical cars, with on average 2.14 L engine size, 8.37 L/100 km and 196 g/km CO₂.
- **Cluster 0** (265 cars): mid-sized cars, with on average 3.02 L, 11.73 L/100 km and 277 g/km CO₂.
- **Cluster 1** (133 cars): large, high-consumption cars, with on average 4.86 L, 14.97 L/100 km and 348 g/km CO₂.
- Cluster 0 does not mean "best": the labels 0, 1 and 2 are arbitrary numbers from k-means and carry no ranking.
- The inverse-scaled cluster centres equal the cluster means, because each k-means centre is the mean of its cars.

## 6. Read a dendrogram

For a readable demonstration we use 15 supplied car rows. Which observations merge early? What does a high merge mean? The cluster cut is provided.

In [ ]:
sample = cars.sample(15, random_state=42)
sample_scaled = scaler.transform(sample[features])
Z = linkage(sample_scaled, method='ward')
names = (sample['make'] + ' ' + sample['model']).tolist()
plt.figure(figsize=(11, 5))
dendrogram(Z, labels=names, leaf_rotation=90)
plt.ylabel('Ward linkage height')
plt.tight_layout()
plt.show()
print('Three-cluster cut:', fcluster(Z, t=3, criterion='maxclust'))

**Answer:**

- **Early merges:** the lowest merges (heights below about 0.3) join nearly identical cars. Examples are the two Toyota RAV4 Hybrid versions (2.5 L, about 6 L/100 km) and the Audi SQ5 with the Range Rover Sport P400 (both 3.0 L, about 11 L/100 km).
- **High merges:** a high merge joins groups that are far apart. The highest merge, at height about 8.6, joins the four economical cars (Corolla Hybrid, Kia K4 and the two RAV4 Hybrids) with all the others.
- **Three-cluster cut:** cutting below the two highest merges (heights 8.6 and 5.1) gives three groups:
  - 4 economical cars;
  - 7 mid-range cars, such as the Audi SQ5, Hyundai Palisade and Jaguar F-Pace;
  - 4 high-consumption cars: Cadillac CT5-V, Porsche 911 GT3, Ram 1500 RHO and INEOS Grenadier.
- This demonstration describes only the 15 sampled cars.

## 7. PCA outlook

The PCA code is supplied. How much variance do PC1 and PC2 retain together? Why is this percentage different from prediction accuracy?

In [ ]:
pca = PCA(n_components=2)
components = pca.fit_transform(scaled)
print('Explained variance ratios:', pca.explained_variance_ratio_.round(3))
print('Total retained:', round(pca.explained_variance_ratio_.sum(), 3))
print('Loadings:')
print(pd.DataFrame(pca.components_, columns=features, index=['PC1', 'PC2']).round(3))
plt.scatter(components[:, 0], components[:, 1], c=labels, cmap='viridis', alpha=0.5)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('PCA projection; colours from the earlier k-means')
plt.show()

**Answer:** PC1 and PC2 together retain 0.897 + 0.100 = 0.997, i.e. 99.7% of the variance of the three scaled inputs.

- **PC1** (89.7%) has almost equal loadings (0.54, 0.60, 0.59). It is an overall "size and consumption" axis: cars with a large engine, high consumption and high CO₂ get high PC1 values.
- **PC2** (10.0%) contrasts engine size (+0.84) with consumption and CO₂ (−0.38, −0.39). Cars with a large engine but relatively low consumption get high PC2 values.
- Nearly all information fits into two axes because consumption and CO₂ move almost exactly together: burning more fuel produces more CO₂.
- Explained variance measures how much input spread remains after projecting, not how many predictions are correct; there is no target here.
- PCA did not use the cluster labels; the colours are added afterwards. The clusters line up mainly along PC1.
- Component signs may be flipped without changing the information.

### Jupyter notebook — footer info

Include this information when submitting your notebook.

In [ ]:
import platform
from datetime import datetime
print(platform.platform())
print("Python:", platform.python_version())
print("Date:", datetime.now().isoformat(timespec="seconds"))